# Electorate Demographics Comparison

This notebook compares ads across electorates with different family income and demographic profiles.

It follows the repo's existing loading pattern, always applies the main-party filter used in `party_analysis.ipynb`, and degrades gracefully if no electorate demographics file is available.

Intended external source:
- DSS Benefit and Payment Recipient Demographics - quarterly data on data.gov.au, which includes electorate-level geography
- If you have a direct CSV/XLSX export from that source, place it under `data/external/` or set the download URL in the cell below


In [ ]:
%load_ext watermark
%watermark -v -n -m -p numpy,pandas,scipy,sklearn,matplotlib,seaborn

import warnings
import os
import sys
import re
from pathlib import Path
from glob import glob
from urllib.request import urlretrieve

warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2
%matplotlib inline

print(f'default sys.path: {sys.path}')

PROJ_ROOT = os.path.abspath(os.path.join(os.pardir))
PROJ_ROOT = os.path.abspath(os.path.join(PROJ_ROOT, os.pardir))
sys.path.append(PROJ_ROOT)
print(f'Project root: {PROJ_ROOT}')

from utils import mpl_settings
from utils.dataset_utilities import load_data
from utils import config

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

mpl_settings.apply_settings()
plt.style.use(mpl_settings.plot_settings)


## Load ads and classify target electorate

`electorate` is the primary field. If it is missing, we fall back to `electorate_from_page_name`.


In [ ]:
file_path = '../../data/processed/2022_aus_elections_mar_to_may.csv'
df = load_data(file_path)

main_parties = ['Liberal', 'Labor', 'Green', 'Independent', 'United Australia Party']
df = df[df['macro_party_uap'].isin(main_parties)].copy()

for col in ['electorate', 'electorate_from_page_name']:
    if col not in df.columns:
        df[col] = pd.NA

primary = df['electorate'].astype('string')
fallback = df['electorate_from_page_name'].astype('string')

has_primary = primary.notna() & primary.str.strip().ne('')
has_fallback = fallback.notna() & fallback.str.strip().ne('')

df['target_electorate'] = primary.where(has_primary, fallback)
df['target_electorate_source'] = np.select(
    [has_primary, has_fallback],
    ['electorate', 'electorate_from_page_name'],
    default='missing'
)

print(f'Total ads after main-party filter: {len(df):,}')
print('Target electorate source counts:')
print(df['target_electorate_source'].value_counts(dropna=False).to_string())
print()
print(f"Missing target electorate share: {df['target_electorate'].isna().mean():.3%}")
print()
print('Top fallback electorates from page name:')
print(df.loc[df['target_electorate_source'].eq('electorate_from_page_name'), 'target_electorate'].value_counts().head(10).to_string())


## Electorate demographics / income data

The notebook looks for a local file first. If no file exists, it creates a placeholder schema so the notebook still runs.

Expected real-data shape:
- `electorate`
- one income column, ideally `median_household_income` or a close variant
- optional demographic columns such as age composition, family composition, or payment recipient shares


In [ ]:
EXTERNAL_DIR = Path('../../data/external')
ELECTORATE_DEMOGRAPHICS_URL = ''  # set this to a direct CSV/XLSX URL if you have one

candidate_patterns = [
    '*electorate*income*.csv',
    '*electorate*demographic*.csv',
    '*dss*electorate*.csv',
    '*commonwealth*electoral*division*.csv',
    '*electorate*.xlsx',
    '*electorate*.xls',
]


def find_local_demographics_file(base_dir, patterns):
    for pattern in patterns:
        matches = sorted(base_dir.glob(pattern))
        if matches:
            return matches[0]
    return None


def read_table(path):
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    raise ValueError(f'Unsupported file type: {path.suffix}')


def normalise_columns(frame):
    frame = frame.copy()
    frame.columns = [str(c).strip().lower() for c in frame.columns]
    rename_map = {
        'division': 'electorate',
        'division_name': 'electorate',
        'electoral_division': 'electorate',
        'electorate_name': 'electorate',
        'name': 'electorate',
        'federal_electorate': 'electorate',
        'median_income': 'median_household_income',
        'median_household_income': 'median_household_income',
        'household_median_income': 'median_household_income',
        'family_income_median': 'median_family_income',
        'median_family_income': 'median_family_income',
        'personal_income_median': 'median_personal_income',
        'median_personal_income': 'median_personal_income',
    }
    frame = frame.rename(columns={k: v for k, v in rename_map.items() if k in frame.columns})
    return frame


def choose_income_column(columns):
    priority = [
        'median_household_income',
        'median_family_income',
        'median_personal_income',
        'household_income',
        'family_income',
        'income',
    ]
    for candidate in priority:
        if candidate in columns:
            return candidate
    income_like = [c for c in columns if 'income' in c]
    return income_like[0] if income_like else None


def maybe_download_file(url, target_dir):
    if not url:
        return None
    target_dir.mkdir(parents=True, exist_ok=True)
    suffix = '.csv'
    target = target_dir / f'electorate_demographics_download{suffix}'
    try:
        urlretrieve(url, target)
        return target
    except Exception as exc:
        print(f'Could not download electorate demographics file: {exc}')
        return None


source_path = find_local_demographics_file(EXTERNAL_DIR, candidate_patterns)
if source_path is None:
    source_path = maybe_download_file(ELECTORATE_DEMOGRAPHICS_URL, EXTERNAL_DIR)

if source_path is not None:
    electorate_profile = normalise_columns(read_table(source_path))
    if 'electorate' not in electorate_profile.columns:
        raise ValueError(f'The file {source_path} does not include an electorate column after normalisation.')
    electorate_profile['electorate'] = electorate_profile['electorate'].astype('string').str.strip()
    electorate_profile = electorate_profile.dropna(subset=['electorate'])
    electorate_profile = electorate_profile.drop_duplicates(subset=['electorate']).copy()
    income_col = choose_income_column(electorate_profile.columns)
    print(f'Loaded electorate demographics from: {source_path}')
    print(f'Income column detected: {income_col}')
    print(f'Available columns: {list(electorate_profile.columns)}')
else:
    print('No local electorate demographics/income file found in ../../data/external.')
    print('Using a placeholder schema with one row per electorate seen in the ads.')
    print('Suggested authoritative source to supply later: DSS Benefit and Payment Recipient Demographics - quarterly data (data.gov.au).')
    electorate_profile = pd.DataFrame({'electorate': sorted(df['target_electorate'].dropna().astype(str).str.strip().unique())})
    electorate_profile['median_household_income'] = pd.NA
    electorate_profile['median_family_income'] = pd.NA
    electorate_profile['median_personal_income'] = pd.NA
    electorate_profile['demographic_profile_source'] = 'placeholder'
    income_col = 'median_household_income'

if income_col is None:
    electorate_profile['median_household_income'] = pd.NA
    income_col = 'median_household_income'

real_income_available = electorate_profile[income_col].notna().sum() > 0
print(f'Real income values available: {real_income_available}')


## Merge ads with electorate profiles and create income tiers


In [ ]:
ad_electorate_df = df[df['target_electorate'].notna()].copy()
ad_electorate_df['target_electorate'] = ad_electorate_df['target_electorate'].astype('string').str.strip()

merged = ad_electorate_df.merge(
    electorate_profile,
    how='left',
    left_on='target_electorate',
    right_on='electorate',
    suffixes=('', '_electorate')
)

print(f'Ads with a target electorate: {len(ad_electorate_df):,}')
print(f'Matched electorate profiles: {merged["electorate"].notna().mean():.3%}')

if 'state' in merged.columns:
    unmatched = merged.loc[merged['electorate'].isna(), 'target_electorate'].value_counts().head(15)
    print()
    print('Top unmatched electorates:')
    print(unmatched.to_string())

# Build income tiers only when we have a real numeric income column.
income_series = pd.to_numeric(merged[income_col], errors='coerce')
merged['median_household_income'] = income_series

income_basis = (
    merged[['target_electorate', 'median_household_income']]
    .dropna()
    .drop_duplicates(subset=['target_electorate'])
    .copy()
)

if income_basis['median_household_income'].nunique() >= 3:
    try:
        income_basis['income_group'] = pd.qcut(
            income_basis['median_household_income'],
            q=3,
            labels=['Low income electorates', 'Mid income electorates', 'High income electorates'],
            duplicates='drop'
        )
    except ValueError:
        income_basis['income_group'] = pd.cut(
            income_basis['median_household_income'],
            bins=3,
            labels=['Low income electorates', 'Mid income electorates', 'High income electorates'],
            include_lowest=True
        )
    income_map = income_basis.set_index('target_electorate')['income_group']
    merged['income_group'] = merged['target_electorate'].map(income_map)
else:
    merged['income_group'] = pd.NA

print()
print('Income group counts:')
print(merged['income_group'].value_counts(dropna=False).to_string())

if merged['income_group'].notna().any():
    income_levels = ['Low income electorates', 'Mid income electorates', 'High income electorates']
    merged['income_group'] = pd.Categorical(merged['income_group'], categories=income_levels, ordered=True)


## Content features and comparison table

The notebook uses simple text features as a portable content baseline: word count, political keyword hits, call-to-action language, and a lightweight propaganda-style proxy.


In [ ]:
keywords_path = Path('../../data/external/keywords.csv')
if keywords_path.exists():
    keywords = pd.read_csv(keywords_path, header=None)[0].dropna().astype(str).str.strip().tolist()
else:
    keywords = ['vote', 'labor', 'liberal', 'greens', 'green', 'election', 'policy', 'cost of living', 'health', 'tax']

keywords = sorted(set(k for k in keywords if k))
keyword_pattern = re.compile(r'(?i)\b(' + '|'.join(re.escape(k) for k in sorted(keywords, key=len, reverse=True)) + r')\b')

cta_terms = [
    'vote', 'join', 'support', 'donate', 'act', 'help', 'sign', 'click', 'share',
    'choose', 'protect', 'stop', 'fight', 'save', 'back'
]
cta_pattern = re.compile(r'(?i)\b(' + '|'.join(re.escape(t) for t in cta_terms) + r')\b')

content_df = merged.copy()
content_df['text'] = content_df['multiword_safe_lemmatized'].fillna('').astype(str)
content_df['ad_word_count'] = content_df['text'].str.split().str.len().fillna(0).astype(int)
content_df['keyword_hits'] = content_df['text'].str.count(keyword_pattern)
content_df['keyword_density'] = np.where(content_df['ad_word_count'] > 0, content_df['keyword_hits'] / content_df['ad_word_count'], np.nan)
content_df['cta_hits'] = content_df['text'].str.count(cta_pattern)
content_df['cta_density'] = np.where(content_df['ad_word_count'] > 0, content_df['cta_hits'] / content_df['ad_word_count'], np.nan)
content_df['propaganda_proxy'] = content_df['keyword_density'].fillna(0) + content_df['cta_density'].fillna(0)

for col in ['ad_delivery_start_time', 'ad_delivery_stop_time']:
    if col in content_df.columns:
        content_df[col] = pd.to_datetime(content_df[col], errors='coerce')

content_df['ad_duration_days'] = (
    (content_df['ad_delivery_stop_time'] - content_df['ad_delivery_start_time']).dt.days.fillna(0).clip(lower=0) + 1
)
content_df['impressions_per_day'] = content_df['mean_impressions'] / content_df['ad_duration_days'].replace(0, np.nan)

summary_cols = [
    'mean_spend', 'mean_impressions', 'ad_duration_days', 'impressions_per_day',
    'ad_word_count', 'keyword_hits', 'keyword_density', 'cta_hits', 'cta_density', 'propaganda_proxy'
]

comparison_table = (
    content_df.dropna(subset=['income_group'])
    .groupby('income_group')[summary_cols]
    .agg(['count', 'mean', 'median'])
)

comparison_table


## Visual comparison


In [ ]:
plot_df = content_df.dropna(subset=['income_group']).copy()

if plot_df.empty:
    print('No real income-group data is available yet, so the comparative plots are skipped.')
    print('Add a real electorate demographics/income file to data/external/ to enable the full analysis.')
else:
    income_order = ['Low income electorates', 'Mid income electorates', 'High income electorates']
    plot_df['income_group'] = pd.Categorical(plot_df['income_group'], categories=income_order, ordered=True)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=False)
    sns.boxplot(data=plot_df, x='income_group', y='mean_spend', ax=axes[0], order=income_order, palette='Set2')
    sns.boxplot(data=plot_df, x='income_group', y='impressions_per_day', ax=axes[1], order=income_order, palette='Set2')
    sns.boxplot(data=plot_df, x='income_group', y='propaganda_proxy', ax=axes[2], order=income_order, palette='Set2')

    axes[0].set_title('Spend by income tier')
    axes[1].set_title('Impressions per day by income tier')
    axes[2].set_title('Content proxy by income tier')

    for ax in axes:
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()

    electorate_level = (
        plot_df.groupby('target_electorate', as_index=False)
        .agg(
            median_household_income=('median_household_income', 'first'),
            mean_spend=('mean_spend', 'mean'),
            mean_impressions=('mean_impressions', 'mean'),
            impressions_per_day=('impressions_per_day', 'mean'),
            propaganda_proxy=('propaganda_proxy', 'mean'),
            n_ads=('id', 'count')
        )
        .dropna(subset=['median_household_income'])
    )

    if not electorate_level.empty:
        fig, ax = plt.subplots(figsize=(9, 6))
        sns.scatterplot(
            data=electorate_level,
            x='median_household_income',
            y='propaganda_proxy',
            size='n_ads',
            sizes=(20, 200),
            hue='mean_spend',
            palette='viridis',
            ax=ax
        )
        ax.set_title('Electorate income vs content proxy')
        ax.set_xlabel('Median household income')
        ax.set_ylabel('Average propaganda proxy')
        plt.tight_layout()
        plt.show()


## Statistical comparison

The main test here is a Kruskal-Wallis comparison across income tiers. When there are enough valid electorates, the notebook also reports a Spearman correlation at electorate level.


In [ ]:
analysis_df = content_df.dropna(subset=['income_group']).copy()

if analysis_df.empty:
    print('No real income tiers are available, so no inferential test can be run yet.')
else:
    income_order = ['Low income electorates', 'Mid income electorates', 'High income electorates']
    test_metrics = ['mean_spend', 'mean_impressions', 'impressions_per_day', 'propaganda_proxy']

    for metric in test_metrics:
        grouped = [analysis_df.loc[analysis_df['income_group'] == grp, metric].dropna() for grp in income_order]
        grouped = [g for g in grouped if len(g) > 0]
        if len(grouped) >= 2:
            stat_value, p_value = stats.kruskal(*grouped)
            print(f'Kruskal-Wallis on {metric}: H={stat_value:.4f}, p={p_value:.4g}')
        else:
            print(f'Not enough non-empty groups for Kruskal-Wallis on {metric}.')

    electorate_level = (
        analysis_df.groupby('target_electorate', as_index=False)
        .agg(
            median_household_income=('median_household_income', 'first'),
            propaganda_proxy=('propaganda_proxy', 'mean'),
            mean_impressions=('mean_impressions', 'mean'),
            mean_spend=('mean_spend', 'mean'),
        )
        .dropna(subset=['median_household_income', 'propaganda_proxy'])
    )

    if len(electorate_level) >= 3:
        rho, rho_p = stats.spearmanr(electorate_level['median_household_income'], electorate_level['propaganda_proxy'])
        print(f'Spearman correlation between median household income and propaganda_proxy: rho={rho:.4f}, p={rho_p:.4g}')
    else:
        print('Not enough electorate-level observations for a Spearman correlation.')

    print()
    print('Income-tier summary (median values):')
    tier_summary = analysis_df.groupby('income_group')[['mean_spend', 'mean_impressions', 'impressions_per_day', 'propaganda_proxy']].median()
    print(tier_summary.to_string())


## Takeaway

If a real electorate income file is available, the notebook will populate low/mid/high income tiers and run the comparison tables and tests above.

If not, it still runs end-to-end and shows the exact columns and file shape you need to supply later.
